# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **$R(s)$**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [ ]:
import numpy as np
from collections import defaultdict

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        # Coordenadas según la imagen (row, col)
        self.start = (0, 0)
        self.walls = {(0, 3), (1, 1), (2, 4), (4, 2)}
        self.slippery_states = {(1, 2), (2, 1), (3, 3)}

        self.terminal_states = {
            (0, 5): 10.0,   # Entrega
            (2, 2): 2.0,    # Carga
            (3, 5): -10.0   # Peligro Mortal
        }

        self.danger_states = {
            (1, 4): -3.0,
            (4, 1): -3.0
        }

        self.living_reward = -1.0
        self.gamma = 0.9

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        r, c = state
        if r < 0 or r >= self.height or c < 0 or c >= self.width:
            return False
        if state in self.walls:
            return False
        return True

    def states(self):
        return [(r, c) for r in range(self.height) for c in range(self.width) if (r, c) not in self.walls]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        if self.is_terminal(state):
            return self.terminal_states[state]
        elif state in self.danger_states:
            return -3.0
        else:
            return self.living_reward

    def get_transition_probs(self, state, action):
        """
        Devuelve:
            [(next_state, probability), ...]
        """
        if self.is_terminal(state):
            return [(state, 1.0)]

        if state in self.slippery_states:
            p_straight, p_left, p_right = 0.60, 0.20, 0.20
        else:
            p_straight, p_left, p_right = 0.90, 0.05, 0.05

        # Calculando las desviaciones relativas a la dirección
        dr, dc = action
        left_action = (-dc, dr)
        right_action = (dc, -dr)

        moves = [
            (action, p_straight),
            (left_action, p_left),
            (right_action, p_right)
        ]

        probs = defaultdict(float)
        for move, p in moves:
            next_r, next_c = state[0] + move[0], state[1] + move[1]
            next_s = (next_r, next_c)
            
            if not self.is_valid_state(next_s):
                next_s = state  # Choca pared/límite y se queda igual
            
            probs[next_s] += p

        return list(probs.items())



### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [ ]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [ ]:
def expected_next_value(grid, state, action, V):
    return sum(p * V[next_s] for next_s, p in grid.get_transition_probs(state, action))

def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    V = {s: 0.0 for s in grid.states()}
    for i in range(max_iter):
        delta = 0
        new_V = {}
        for s in grid.states():
            if grid.is_terminal(s):
                new_V[s] = grid.get_reward(s)
            else:
                max_v = max(grid.get_reward(s) + grid.gamma * expected_next_value(grid, s, a, V) for a in grid.actions)
                new_V[s] = max_v
            delta = max(delta, abs(V[s] - new_V[s]))
        V = new_V
        if delta < threshold:
            return V, i + 1
    return V, max_iter

def extract_policy(grid, V):
    policy = {}
    for s in grid.states():
        if grid.is_terminal(s):
            policy[s] = None
            continue
        best_a = None
        best_v = float('-inf')
        for a in grid.actions:
            val = expected_next_value(grid, s, a, V)
            if val > best_v:
                best_v = val
                best_a = a
        policy[s] = best_a
    return policy



## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [ ]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    V = {s: 0.0 for s in grid.states()}
    for i in range(max_iter):
        delta = 0
        new_V = {}
        for s in grid.states():
            if grid.is_terminal(s):
                new_V[s] = grid.get_reward(s)
            else:
                a = policy[s]
                new_V[s] = grid.get_reward(s) + grid.gamma * expected_next_value(grid, s, a, V)
            delta = max(delta, abs(V[s] - new_V[s]))
        V = new_V
        if delta < threshold:
            break
    return V

def policy_improvement(grid, V):
    return extract_policy(grid, V)

def policy_iteration(grid, threshold=1e-4, max_iter=100):
    # 1. política inicial arbitraria
    policy = {s: grid.actions[0] for s in grid.states() if not grid.is_terminal(s)}
    for s in grid.states():
        if grid.is_terminal(s):
            policy[s] = None

    history = 0
    for i in range(max_iter):
        history += 1
        # 2. evaluación
        V = policy_evaluation(grid, policy, threshold)
        
        # 3. mejora
        new_policy = policy_improvement(grid, V)
        
        # 4. repetir hasta estabilidad
        stable = True
        for s in grid.states():
            if not grid.is_terminal(s):
                if policy[s] != new_policy[s]:
                    stable = False
                    break
        policy = new_policy
        
        if stable:
            break
            
    return policy, V, history



## Parte 4 — Visualización y comparación


In [ ]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))

def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [ ]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")



## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?
2. ¿Por qué una recompensa menor podría ser óptima?
3. ¿En qué estados el piso resbaloso cambia la decisión?
4. ¿Qué papel cumple el costo por paso `-1`?
5. ¿Por qué $T(s,a,s')$ ya no puede implementarse con las mismas probabilidades para todos los estados?

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.


### Respuestas y análisis a la Parte 5

1. **¿El robot busca la entrega o la estación de carga?**
   Depende estrechamente de la recompensa por paso `living_reward`. Con un castigo agresivo de `-1` por cada paso transitado y con los peligros distribuidos (resbalosos y `danger -3`), la gran distancia hacia la entrega `+10` desde la posición `(0,0)` diluye su valor drásticamente (el factor de descuento `gamma=0.9` entra en acción, más el `-1` constante). Por ende, en la configuración actual la política apuntará hacia el `+2` que está considerablemente más cerca, evitando acumular más penalizaciones.

2. **¿Por qué una recompensa menor podría ser óptima?**
   Por el principio de utilidad temporal y riesgo. Un camino más corto con menos pasos acumula menos castigos de `living_reward` y disminuye la probabilidad de terminar por accidente en un estado de `-3` o `-10` debido a la transición estocástica del piso normal/resbaloso. La recompensa a largo plazo sufre mucho impacto por el riesgo de navegación.

3. **¿En qué estados el piso resbaloso cambia la decisión?**
   Particularmente en las celdas adyacentes a peligros o donde el corredor es estrecho (como la celda `(1,2)` que pasa al lado del peligro o el terminal y `(3,3)`). En estas casillas, el robot podría intentar chocar intencionalmente contra la pared con la acción principal para usar las "desviaciones" del 20% como un medio de movimiento más seguro que apuntar directamente al espacio abierto donde una desviación accidental lo tiraría al peligro.

4. **¿Qué papel cumple el costo por paso `-1`?**
   Actúa como una "presión de tiempo". Incentiva al agente a encontrar un estado terminal (para acabar el episodio) de la forma más rápida y directa posible en vez de deambular dando vueltas o chocando contra paredes para maximizar probabilidades.

5. **¿Por qué $T(s,a,s')$ ya no puede implementarse con las mismas probabilidades para todos los estados?**
   Porque el entorno es intrínsecamente heterogéneo. El piso no es unificado: en ciertas baldosas la estocasticidad se intensifica drásticamente ($0.60/0.20/0.20$ vs $0.90/0.05/0.05$), forzando a que la función de probabilidad de transición dependa estrictamente del estado actual (`s`) y no solo de la acción ejecutada.

---
**Sobre los Experimentos:**

- **A (Menos costo por paso = -0.1):** Predicción: Al ser mucho más barato caminar por la cuadrícula, el agente se atreverá a tomar el camino largo asumiendo un poco más de riesgo para capturar el glorioso `+10`, ignorando el `+2` de carga.
- **B (Piso muy resbaloso = 0.40):** El riesgo de terminar en `Danger` o no llegar a la meta aumenta exponencialmente. El robot probablemente intentará alejarse del todo de estas zonas o usar un camino que utilice paredes como "parachoques" para absorber los desvíos probabilísticos fallidos.
- **C (Más paciencia gamma = 0.99):** Sí. Una recompensa tardía mantiene un altísimo valor temporal. Combinado con un bajo living reward, el agente está dispuesto a caminar grandes distancias si la recompensa supera por mucho a las recompensas cortas, persiguiendo `+10`.